# Time-Series Feature Engineering

Construct leakage-safe lag, rolling, calendar, holiday, and store-level features for sales forecasting.

In [2]:
from pathlib import Path

import pandas as pd
import numpy as np

In [3]:
def find_project_root():
    path = Path.cwd().resolve()

    for candidate in [path, *path.parents]:
        if (candidate / "data" / "raw").exists():
            return candidate

    raise FileNotFoundError("Project root not found")


PROJECT_ROOT = find_project_root()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

In [4]:
store_week = pd.read_csv(
    PROCESSED_DIR / "store_week.csv"
)

store_week["Date"] = pd.to_datetime(
    store_week["Date"]
)

## Source Dataset Validation

In [6]:
store_week = (
    store_week
    .sort_values(["Store", "Date"])
    .reset_index(drop=True)
)

assert store_week["Store"].nunique() == 45
assert store_week["Date"].nunique() == 143

assert not store_week.duplicated(
    ["Store", "Date"]
).any()

print("Source dataset validated.")

Source dataset validated.


In [7]:
date_gaps = (
    store_week
    .groupby("Store")["Date"]
    .diff()
    .dropna()
)

assert (date_gaps == pd.Timedelta(days=7)).all()

print("Weekly continuity validated.")

Weekly continuity validated.


## Calendar Features

In [9]:
store_week["WeekOfYear"] = (
    store_week["Date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

store_week["Month"] = (
    store_week["Date"].dt.month
)

store_week["Quarter"] = (
    store_week["Date"].dt.quarter
)

store_week["Year"] = (
    store_week["Date"].dt.year
)

In [10]:
assert store_week["WeekOfYear"].between(1, 53).all()
assert store_week["Month"].between(1, 12).all()
assert store_week["Quarter"].between(1, 4).all()

print("Calendar features validated.")

Calendar features validated.


## Holiday Features

Use known holiday dates as calendar information available before forecasting.

In [12]:
holiday_dates = {
    "2010-02-12": "Super Bowl",
    "2010-09-10": "Labor Day",
    "2010-11-26": "Thanksgiving",
    "2010-12-31": "Christmas",

    "2011-02-11": "Super Bowl",
    "2011-09-09": "Labor Day",
    "2011-11-25": "Thanksgiving",
    "2011-12-30": "Christmas",

    "2012-02-10": "Super Bowl",
    "2012-09-07": "Labor Day",
    "2012-11-23": "Thanksgiving",
    "2012-12-28": "Christmas"
}

holiday_dates = {
    pd.Timestamp(date): name
    for date, name in holiday_dates.items()
}

In [13]:
store_week["HolidayName"] = (
    store_week["Date"]
    .map(holiday_dates)
    .fillna("None")
)

In [14]:
holiday_counts = (
    store_week["HolidayName"]
    .value_counts()
    .sort_index()
)

holiday_counts

HolidayName
Christmas         90
Labor Day        135
None            5985
Super Bowl       135
Thanksgiving      90
Name: count, dtype: int64

## Lag Features

Lag variables use historical sales from the same store only.

- `lag_1`: previous week
- `lag_2`: two weeks prior
- `lag_52`: approximately the same week one year earlier

In [16]:
store_week["lag_1"] = (
    store_week
    .groupby("Store")["Weekly_Sales"]
    .shift(1)
)

In [17]:
store_week["lag_2"] = (
    store_week
    .groupby("Store")["Weekly_Sales"]
    .shift(2)
)

In [18]:
store_week["lag_52"] = (
    store_week
    .groupby("Store")["Weekly_Sales"]
    .shift(52)
)

In [19]:
store_1 = (
    store_week[
        store_week["Store"] == store_week["Store"].iloc[0]
    ]
    .sort_values("Date")
    .reset_index(drop=True)
)

assert (
    store_1.loc[1, "lag_1"]
    == store_1.loc[0, "Weekly_Sales"]
)

assert (
    store_1.loc[2, "lag_2"]
    == store_1.loc[0, "Weekly_Sales"]
)

assert (
    store_1.loc[52, "lag_52"]
    == store_1.loc[0, "Weekly_Sales"]
)

print("Lag features validated.")

Lag features validated.


## Rolling Sales Features

Rolling statistics are calculated after shifting sales by one week.

This prevents the current target from entering its own predictors.

In [21]:
historical_sales = (
    store_week
    .groupby("Store")["Weekly_Sales"]
    .shift(1)
)

In [22]:
store_week["rolling_mean_4"] = (
    historical_sales
    .groupby(store_week["Store"])
    .rolling(4)
    .mean()
    .reset_index(level=0, drop=True)
)

In [23]:
store_week["rolling_std_4"] = (
    historical_sales
    .groupby(store_week["Store"])
    .rolling(4)
    .std()
    .reset_index(level=0, drop=True)
)

In [24]:
check_store = (
    store_week[
        store_week["Store"] == store_week["Store"].iloc[0]
    ]
    .sort_values("Date")
    .reset_index(drop=True)
)

expected_mean = (
    check_store.loc[0:3, "Weekly_Sales"]
    .mean()
)

assert pd.isna(
    check_store.loc[3, "rolling_mean_4"]
)

assert np.isclose(
    check_store.loc[4, "rolling_mean_4"],
    expected_mean
)

print("Rolling features validated.")

Rolling features validated.


## Store-Level Features

Store type and size provide persistent structural information about demand.

In [26]:
store_attributes = (
    store_week[
        ["Store", "Type", "Size"]
    ]
    .drop_duplicates("Store")
)

assert len(store_attributes) == 45

In [27]:
assert (
    store_attributes["Store"].nunique()
    == 45
)

assert (
    store_attributes["Type"].notna().all()
)

assert (
    store_attributes["Size"].notna().all()
)

print("Store attributes validated.")

Store attributes validated.


## Candidate External Features

Available external variables are retained for controlled evaluation.

They are not automatically included in the final model.

In [29]:
external_features = [
    "Temperature",
    "Fuel_Price",
    "MarkDown1",
    "MarkDown2",
    "MarkDown3",
    "MarkDown4",
    "MarkDown5",
    "CPI",
    "Unemployment"
]

available_external = [
    col
    for col in external_features
    if col in store_week.columns
]

print("Available external features:")
print(available_external)

Available external features:
['Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment']


## Core Modeling Features

The initial forecasting feature set focuses on:

- Historical sales
- Calendar structure
- Holiday information
- Store attributes

External variables remain candidates for controlled comparison.

In [31]:
target = "Weekly_Sales"

feature_columns = [
    "lag_1",
    "lag_2",
    "lag_52",
    "rolling_mean_4",
    "rolling_std_4",
    "WeekOfYear",
    "Month",
    "HolidayName",
    "Type",
    "Size"
]

In [32]:
missing_features = [
    col
    for col in feature_columns
    if col not in store_week.columns
]

assert not missing_features

print("Core feature set validated.")

Core feature set validated.


## Modeling Dataset

`lag_52` requires one year of historical sales.

Rows without the required historical information are excluded from the modeling dataset.

In [34]:
model_data = store_week.copy()

required_columns = [
    target,
    *feature_columns
]

model_data = model_data.dropna(
    subset=required_columns
).reset_index(drop=True)

In [35]:
print("Rows:", len(model_data))
print("Stores:", model_data["Store"].nunique())
print(
    "Date range:",
    model_data["Date"].min().date(),
    "to",
    model_data["Date"].max().date()
)

Rows: 4095
Stores: 45
Date range: 2011-02-04 to 2012-10-26


## Leakage Checks

In [37]:
for store_id, group in model_data.groupby("Store"):
    dates = group["Date"].sort_values()

    assert dates.is_monotonic_increasing
    assert dates.diff().dropna().eq(
        pd.Timedelta(days=7)
    ).all()

print("Chronological ordering validated within stores.")

Chronological ordering validated within stores.


In [38]:
for store_id, group in model_data.groupby("Store"):
    group = group.sort_values("Date").reset_index(drop=True)

    if len(group) >= 5:
        expected = group.loc[
            0:3, "Weekly_Sales"
        ].mean()

        assert np.isclose(
            group.loc[4, "rolling_mean_4"],
            expected
        )

print("Rolling features use only prior observations.")

Rolling features use only prior observations.


## Final Feature Validation

In [40]:
feature_missing = (
    model_data[required_columns]
    .isna()
    .sum()
)

feature_missing

Weekly_Sales      0
lag_1             0
lag_2             0
lag_52            0
rolling_mean_4    0
rolling_std_4     0
WeekOfYear        0
Month             0
HolidayName       0
Type              0
Size              0
dtype: int64

In [41]:
assert (
    model_data[required_columns]
    .isna()
    .sum()
    .sum()
    == 0
)

assert not model_data.duplicated(
    ["Store", "Date"]
).any()

assert model_data["Store"].nunique() == 45

print("Final feature validation passed.")

Final feature validation passed.


## Save Modeling Dataset

In [43]:
output_path = (
    PROCESSED_DIR / "store_week_features.csv"
)

model_data.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path.name)

Saved: store_week_features.csv


In [44]:
print("Final dataset shape:", model_data.shape)
print("Stores:", model_data["Store"].nunique())
print("Core modeling features:", len(feature_columns))
print(
    "Date range:",
    model_data["Date"].min().date(),
    "to",
    model_data["Date"].max().date()
)
print("Output:", output_path.name)

Final dataset shape: (4095, 25)
Stores: 45
Core modeling features: 10
Date range: 2011-02-04 to 2012-10-26
Output: store_week_features.csv


## Feature Summary

| Feature | Purpose |
|---|---|
| `lag_1` | Recent weekly demand |
| `lag_2` | Short-term demand history |
| `lag_52` | Year-over-year seasonal signal |
| `rolling_mean_4` | Recent demand level |
| `rolling_std_4` | Recent demand variability |
| `WeekOfYear` | Annual seasonality |
| `Month` | Calendar seasonality |
| `HolidayName` | Holiday effects |
| `Type` | Store structure |
| `Size` | Store scale |

## Key Findings

- 52-week history is required for the year-over-year lag.
- The resulting modeling dataset contains 4,095 Store-Week observations.
- Lag features are calculated within each store.
- Rolling statistics use only prior observations.
- Calendar and holiday variables provide known future information.
- External variables are retained as candidates rather than assumed to improve forecasting.
- The feature pipeline is validated for Store-Week uniqueness and historical leakage.